# 🏈 NFL Draft Scouting Alpha — NLP Classification Pipeline

> **Goal:** Find the *Hidden Delta* between a scout's numerical grade and their text.
> Identify the *Linguistic Success Centroid* for each position group and classify every
> player into one of four profile buckets: **Clean Hit**, **Hidden Gem**, **Grade-Text Mismatch (Overrated)**,
> or **Hedge Risk**.

| Section | Content |
|---------|---------|
| 1 | Data Loading & Preprocessing |
| 2 | Keyword Taxonomy Definition |
| 3 | TF-IDF Vectorization & Cosine Similarity |
| 4 | Top 5 Success / Bust Anchor Words Per Position |
| 5 | Mismatch Profile Detection |
| 6 | Predictive Power Analysis |
| 7 | Output Summary |

**Training window:** 2014–2021 NFL Drafts (reliable second-contract labels).
**Data source:** `data/processed/draft_enriched_with_contracts.csv`


## Section 1 — Data Loading & Preprocessing

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Viz
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Download required NLTK data (silent if already present)
for pkg in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

print("✅ Imports complete")


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjust ROOT if running from a different working directory.
ROOT = Path('..') if Path('..').joinpath('data').exists() else Path('.')
DATA_PATH = ROOT / 'data' / 'processed' / 'draft_enriched_with_contracts.csv'

print(f"Data path: {DATA_PATH.resolve()}")
print(f"Exists: {DATA_PATH.exists()}")


In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw shape: {raw.shape}")
print(f"Years available: {sorted(raw['year'].unique())}")
print(f"\nColumns:\n{list(raw.columns)}")


In [ ]:
# ── Filter to training window: 2014–2021 ──────────────────────────────────────
# Only keep rows that have scouting text, a grade, and a contract outcome label.
df = raw.query('2014 <= year <= 2021').copy()

text_cols = ['overview', 'strengths', 'weaknesses']
df[text_cols] = df[text_cols].fillna('')

# Combine sections into a single scouting text field
df['scouting_text'] = (
    df[text_cols].agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Cast grade to numeric
df['grade'] = pd.to_numeric(df['grade'], errors='coerce')

# Cast made_it_contract to int (True→1, False→0)
df['made_it_contract'] = df['made_it_contract'].map({True: 1, False: 0, 1: 1, 0: 0})

# Drop rows missing the key fields
df = df.dropna(subset=['scouting_text', 'grade', 'made_it_contract', 'Pos_Group'])
df = df[df['scouting_text'].str.strip() != '']
df = df.reset_index(drop=True)

print(f"Training set: {df.shape[0]} players, {df['year'].nunique()} draft classes")
print(f"\nOutcome distribution:")
print(df['made_it_contract'].value_counts().rename({1: 'Earned 2nd contract', 0: 'Did not'}).to_string())
print(f"\nPosition group counts:")
print(df['Pos_Group'].value_counts().to_string())


In [ ]:
# ── Position group mapping ────────────────────────────────────────────────────
# The analysis uses these six position groups (matching the data's Pos_Group column).
# Note: 'CB' in the spec maps to 'DB' here (DBs include CBs and safeties).
#       'OT' maps to 'OL' (all offensive linemen).
TARGET_POSITIONS = ['EDGE', 'OL', 'WR', 'QB', 'RB', 'DB']

# If you want to narrow to strict CB/OT only, uncomment:
# df['Pos_Group_strict'] = df['position'].apply(
#     lambda p: 'CB' if 'CB' in str(p) else ('OT' if p in ('OT','T') else df.loc[...,'Pos_Group'])
# )

pos_df = df[df['Pos_Group'].isin(TARGET_POSITIONS)].copy()
print(f"Filtered to target positions: {pos_df.shape[0]} players")
print(pos_df['Pos_Group'].value_counts().to_string())


In [ ]:
# ── NFL-Aware Preprocessing (adapted from legacy_annual_tf_idf.ipynb) ─────────
#
# Key design decisions (see legacy notebook for full rationale):
#   1. KEEP_WORDS: directional adjectives NLTK would remove that carry scouting signal
#      e.g. "high" (pad level), "low" (leverage), "quick" (first step)
#   2. CUSTOM_STOPS: generic scouting filler with near-zero discriminative value
#   3. PHRASE_BLOCKLIST: outcome-leaking phrases removed BEFORE tokenization
#   4. Hyphen normalization: "hand-fighting" → "hand fighting" (captured as bigram)
#   5. Lemmatization (WordNet): "fighting"→"fight", "routes"→"route"

KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'work', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type',
}

# Outcome-leaking phrases: remove before any processing
PHRASE_BLOCKLIST = [
    'undrafted free agent', 'practice squad', 'free agent', 'early starter',
    'pro bowl', 'late round', 'undrafted free', 'make roster', 'rostered',
]

_base_stops = set(stopwords.words('english'))
NFL_STOPWORDS = (_base_stops - KEEP_WORDS) | CUSTOM_STOPS
_lemmatizer = WordNetLemmatizer()


def nfl_preprocess(text: str) -> str:
    """NFL-aware text preprocessing pipeline.

    Steps:
      1. Lowercase
      2. Strip outcome-leaking phrases (PHRASE_BLOCKLIST)
      3. Normalize hyphens → space (preserves bigram collocations)
      4. Remove non-alpha characters
      5. Tokenize, apply NFL_STOPWORDS, min length filter
      6. WordNet lemmatization
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    # Remove outcome-leaking phrases first (longest first to avoid partial matches)
    for phrase in sorted(PHRASE_BLOCKLIST, key=len, reverse=True):
        text = text.replace(phrase, ' ')
    text = re.sub(r'[-\u2013\u2014]', ' ', text)   # hyphens → space
    text = re.sub(r'[^a-z\s]', ' ', text)           # letters + spaces only
    tokens = text.split()
    tokens = [t for t in tokens if t not in NFL_STOPWORDS and len(t) > 1]
    tokens = [_lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)


pos_df['clean_text'] = pos_df['scouting_text'].apply(nfl_preprocess)
pos_df = pos_df[pos_df['clean_text'].str.strip() != ''].copy()
pos_df = pos_df.reset_index(drop=True)

token_counts = pos_df['clean_text'].str.split().str.len()
print(f"Players after preprocessing: {len(pos_df)}")
print(f"Tokens/player — median: {int(token_counts.median())}, "
      f"mean: {token_counts.mean():.0f}, "
      f"min: {token_counts.min()}, max: {token_counts.max()}")
print(f"\nSample (200 chars):")
print(pos_df['clean_text'].iloc[0][:200])


## Section 2 — Keyword Taxonomy Definition

Four vocabulary clusters capture distinct scouting signal types:

| Cluster | Signal type | Interpretation |
|---------|-------------|---------------|
| **Raw Athlete** | Physical upside | High bust rate — scouts hedging on developmental players |
| **Technique** | Football-specific skills | Alpha signal — process-driven scouts identifying real contributors |
| **Hedge Language** | Conditional, uncertain | Risk flag — scouts softening opinions |
| **Definitive Language** | Confident, declarative | Success signal — scouts committing to player's current ability |


In [ ]:
# ── Keyword Taxonomy ──────────────────────────────────────────────────────────
# These lists are designed to survive lemmatization (WordNet lemmatizes to base forms).
# Multi-word phrases are matched via substring search (not TF-IDF tokens).

RAW_ATHLETE_TERMS = [
    'trait', 'upside', 'frame', 'basketball background', 'athleticism',
    'raw', 'potential', 'tool', 'developmental',
]

TECHNIQUE_TERMS = [
    'hand fight', 'leverage', 'nuance', 'landmark', 'stack',
    'footwork', 'pad level', 'anchor', 'technique', 'assignment',
]

HEDGE_TERMS = [
    'but', 'if', 'when he', 'flash', 'could be', 'ha the potential',
    'projection', 'once he', 'need to',
]

DEFINITIVE_TERMS = [
    'will', 'doe', 'consistently', 'reliable', 'proven',
    'demonstrate', 'finish', 'control',
]

# Store in dict for easy iteration
TAXONOMY = {
    'raw_athlete': RAW_ATHLETE_TERMS,
    'technique': TECHNIQUE_TERMS,
    'hedge': HEDGE_TERMS,
    'definitive': DEFINITIVE_TERMS,
}

print("Keyword Taxonomy (post-lemmatization compatible):")
for cluster, terms in TAXONOMY.items():
    print(f"  {cluster:12s}: {terms}")


In [ ]:
# ── Compute keyword densities ─────────────────────────────────────────────────
# Density = (number of keyword matches) / (total tokens in clean_text)
# Matches are counted on the *cleaned* (lemmatized) text so keyword lists
# must use base/lemma forms (e.g. "trait" not "traits", "flash" not "flashes").

def keyword_density(text: str, terms: list) -> float:
    """Fraction of text tokens that match any keyword in the taxonomy cluster."""
    if not text or not text.strip():
        return 0.0
    tokens = text.split()
    n = len(tokens)
    if n == 0:
        return 0.0
    # Count substring matches (handles multi-word phrases)
    matches = sum(1 for t in tokens if any(kw in t for kw in terms))
    # Also check phrase matches on full text for multi-word keywords
    multi_word = [kw for kw in terms if ' ' in kw]
    phrase_matches = sum(text.count(kw) for kw in multi_word)
    return (matches + phrase_matches) / n


for col, terms in TAXONOMY.items():
    pos_df[f'{col}_density'] = pos_df['clean_text'].apply(
        lambda t: keyword_density(t, terms)
    )

density_cols = [f'{c}_density' for c in TAXONOMY]
print("Keyword density stats (mean by position group):")
display_df = pos_df.groupby('Pos_Group')[density_cols].mean().round(4)
display_df.columns = [c.replace('_density', '') for c in display_df.columns]
print(display_df.to_string())


## Section 3 — TF-IDF Vectorization & Cosine Similarity

For each position group:
1. Fit a single TF-IDF vectorizer on all reports (unigrams + bigrams)
2. Compute the **Success Centroid** = mean TF-IDF vector of `made_it_contract=1` players
3. Compute the **Bust Centroid** = mean TF-IDF vector of `made_it_contract=0` players
4. Compute each player's cosine similarity to both centroids

**TF-IDF parameters (inherited from legacy pipeline):**
- `ngram_range=(1, 2)` — captures collocations like "pad level", "pass rush"
- `max_features=1500` — vocabulary cap to reduce noise
- `min_df=3` — ignore terms appearing in < 3 documents (hapax / typos)
- `sublinear_tf=True` — log-scale term frequency (stabilises short docs)


In [ ]:
# ── TF-IDF vectorization per position group ───────────────────────────────────
TFIDF_PARAMS = dict(
    max_features=1500,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True,
)

pos_results = {}   # position → dict of vectorizer, matrix, centroids, vocab

for pos in TARGET_POSITIONS:
    sub = pos_df[pos_df['Pos_Group'] == pos].copy()
    if len(sub) < 20:
        print(f"⚠️  {pos}: only {len(sub)} players — skipping (too few samples)")
        continue

    vec = TfidfVectorizer(**TFIDF_PARAMS)
    X = vec.fit_transform(sub['clean_text'])
    vocab = np.array(vec.get_feature_names_out())

    hit_mask = sub['made_it_contract'].values == 1
    bust_mask = ~hit_mask

    hit_centroid  = X[hit_mask].mean(axis=0).A1  if hit_mask.sum()  > 0 else np.zeros(X.shape[1])
    bust_centroid = X[bust_mask].mean(axis=0).A1 if bust_mask.sum() > 0 else np.zeros(X.shape[1])

    # Cosine similarity: each player vs. both centroids
    sim_hit  = cosine_similarity(X, hit_centroid.reshape(1, -1)).ravel()
    sim_bust = cosine_similarity(X, bust_centroid.reshape(1, -1)).ravel()

    pos_results[pos] = {
        'sub': sub.copy().reset_index(drop=True),
        'vectorizer': vec,
        'X': X,
        'vocab': vocab,
        'hit_centroid': hit_centroid,
        'bust_centroid': bust_centroid,
        'sim_hit': sim_hit,
        'sim_bust': sim_bust,
        'n_hits': hit_mask.sum(),
        'n_busts': bust_mask.sum(),
    }

    print(f"✅ {pos:4s}: {len(sub):4d} players | "
          f"{hit_mask.sum()} hits | {bust_mask.sum()} busts | "
          f"vocab {len(vocab)} terms")


In [ ]:
# ── Attach cosine similarities back to pos_df ─────────────────────────────────
pos_df['sim_hit']  = np.nan
pos_df['sim_bust'] = np.nan

for pos, res in pos_results.items():
    idx = res['sub'].index
    pos_df.loc[pos_df['Pos_Group'] == pos, 'sim_hit']  = res['sim_hit']
    pos_df.loc[pos_df['Pos_Group'] == pos, 'sim_bust']  = res['sim_bust']

print("Cosine similarity to Success Centroid by position group:")
print(pos_df.groupby('Pos_Group')[['sim_hit', 'sim_bust']].mean().round(4).to_string())


In [ ]:
# ── Centroid similarity distributions ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
axes = axes.ravel()

for i, pos in enumerate(TARGET_POSITIONS):
    if pos not in pos_results:
        continue
    ax = axes[i]
    res = pos_results[pos]
    sub = res['sub']
    hit_sims  = res['sim_hit'][sub['made_it_contract'].values == 1]
    bust_sims = res['sim_hit'][sub['made_it_contract'].values == 0]

    ax.hist(bust_sims, bins=25, alpha=0.6, color='tomato', label='Bust (no 2nd contract)', density=True)
    ax.hist(hit_sims,  bins=25, alpha=0.6, color='steelblue', label='Hit (earned 2nd contract)', density=True)
    ax.set_title(f'{pos} — Cosine Sim to Success Centroid', fontweight='bold')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.axvline(0.40, color='gold', linestyle='--', lw=1.5, label='0.40 threshold')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Cosine Similarity to Success Centroid by Position',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('centroid_similarity_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: centroid_similarity_distributions.png")


## Section 4 — Top 5 Success / Bust Anchor Words Per Position

**Method: Log-Odds Ratio**

For each term in the vocabulary:

$$\text{log-odds}(w) = \log\!\left(\frac{P(w \mid \text{hit}) + \epsilon}{P(w \mid \text{bust}) + \epsilon}\right)$$

- High positive log-odds → word disproportionately appears in *hit* reports
- High negative log-odds → word disproportionately appears in *bust* reports

Smoothing constant ε = 1e-9 avoids division-by-zero on rare terms.


In [ ]:
# ── Log-odds anchor words per position ────────────────────────────────────────
EPSILON = 1e-9
TOP_N = 5
anchor_rows = []

print(f"{'POS':4s}  {'RANK':4s}  {'SUCCESS WORD':25s}  {'LOG-ODDS':>8s}    {'BUST WORD':25s}  {'LOG-ODDS':>8s}")
print("-" * 90)

for pos in TARGET_POSITIONS:
    if pos not in pos_results:
        continue
    res = pos_results[pos]
    sub = res['sub']
    X = res['X']
    vocab = res['vocab']

    hit_mask  = sub['made_it_contract'].values == 1
    bust_mask = ~hit_mask

    # Mean TF-IDF per group (serves as term probability proxy)
    hit_mean  = X[hit_mask].mean(axis=0).A1  if hit_mask.sum()  > 0 else np.zeros(X.shape[1])
    bust_mean = X[bust_mask].mean(axis=0).A1 if bust_mask.sum() > 0 else np.zeros(X.shape[1])

    log_odds = np.log((hit_mean + EPSILON) / (bust_mean + EPSILON))

    top_success_idx = np.argsort(log_odds)[::-1][:TOP_N]
    top_bust_idx    = np.argsort(log_odds)[:TOP_N]

    for rank in range(TOP_N):
        s_idx = top_success_idx[rank]
        b_idx = top_bust_idx[rank]
        anchor_rows.append({
            'pos_group': pos,
            'rank': rank + 1,
            'success_word': vocab[s_idx],
            'success_log_odds': round(log_odds[s_idx], 4),
            'bust_word': vocab[b_idx],
            'bust_log_odds': round(log_odds[b_idx], 4),
        })
        print(f"{pos:4s}  {rank+1:4d}  {vocab[s_idx]:25s}  {log_odds[s_idx]:+8.4f}    "
              f"{vocab[b_idx]:25s}  {log_odds[b_idx]:+8.4f}")
    print()

anchor_df = pd.DataFrame(anchor_rows)


In [ ]:
# ── Anchor word heatmap visualization ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, word_type, col_lo, color in [
    (axes[0], 'success', 'success_log_odds', 'steelblue'),
    (axes[1], 'bust',    'bust_log_odds',    'tomato'),
]:
    word_col = f'{word_type}_word'
    heat_data = anchor_df.pivot(index=word_col, columns='pos_group', values=col_lo).fillna(0)
    # Sort by mean absolute value
    heat_data = heat_data.loc[heat_data.abs().mean(axis=1).sort_values(ascending=False).index]

    sns.heatmap(
        heat_data,
        ax=ax,
        cmap='RdYlGn' if word_type == 'success' else 'RdYlGn_r',
        center=0,
        annot=True,
        fmt='.2f',
        linewidths=0.5,
        cbar_kws={'label': 'Log-Odds'},
    )
    ax.set_title(f'Top {TOP_N} {"Success" if word_type=="success" else "Bust"} Anchor Words\n(Log-Odds Ratio)',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Position Group')
    ax.set_ylabel('Term')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('anchor_word_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: anchor_word_heatmap.png")


In [ ]:
# ── Clean summary table ───────────────────────────────────────────────────────
summary_pivot = anchor_df.groupby('pos_group').apply(
    lambda g: pd.Series({
        'Top 5 Success Words': ', '.join(g.sort_values('rank')['success_word'].tolist()),
        'Top 5 Bust Words':    ', '.join(g.sort_values('rank')['bust_word'].tolist()),
    })
).reset_index().rename(columns={'pos_group': 'Position Group'})

print("\nAnchor Word Summary Table:")
print(summary_pivot.to_string(index=False))


## Section 5 — Mismatch Profile Detection

Each player is assigned one of four profile labels:

| Profile | Condition | Interpretation |
|---------|-----------|---------------|
| **Grade-Text Mismatch (Overrated)** | Grade > 6.5 AND sim_hit < 0.40 | Scout's words don't back up the grade |
| **Hidden Gem** | Grade 6.0–6.3 AND technique_density > median AND sim_hit > 0.45 | Under-graded players with elite process language |
| **Hedge Risk** | Grade ≥ 7.0 AND hedge_density > position median | Top-graded players surrounded by conditional language |
| **Clean Hit** | sim_hit in top quartile AND grade ≥ 6.5 AND hedge_density ≤ median | High grade backed by confident, process-rich text |

Priority: Overrated > Hidden Gem > Hedge Risk > Clean Hit > Unclassified


In [ ]:
# ── Compute per-position medians for hedge density ────────────────────────────
pos_medians = pos_df.groupby('Pos_Group')['hedge_density'].median().rename('hedge_median')
pos_df = pos_df.merge(pos_medians, on='Pos_Group', how='left')

# ── Assign mismatch profiles ──────────────────────────────────────────────────
def assign_profile(row):
    grade     = row['grade']
    sim_hit   = row['sim_hit']
    tech_d    = row['technique_density']
    hedge_d   = row['hedge_density']
    hedge_med = row['hedge_median']

    # Order matters: more specific / extreme conditions first
    if grade > 6.5 and (pd.isna(sim_hit) or sim_hit < 0.40):
        return 'Grade-Text Mismatch (Overrated)'
    if 6.0 <= grade <= 6.3 and not pd.isna(sim_hit) and sim_hit > 0.45:
        return 'Hidden Gem'
    if grade >= 7.0 and hedge_d > hedge_med:
        return 'Hedge Risk'
    if not pd.isna(sim_hit) and sim_hit > pos_df['sim_hit'].quantile(0.75) and grade >= 6.5 and hedge_d <= hedge_med:
        return 'Clean Hit'
    return 'Unclassified'

pos_df['profile'] = pos_df.apply(assign_profile, axis=1)

profile_counts = pos_df['profile'].value_counts()
print("Profile distribution:")
print(profile_counts.to_string())
print()
print("Profile by position group:")
print(pd.crosstab(pos_df['Pos_Group'], pos_df['profile']).to_string())


In [ ]:
# ── Profile × actual outcome ──────────────────────────────────────────────────
hit_rate = pos_df.groupby('profile')['made_it_contract'].agg(['mean', 'count'])
hit_rate.columns = ['Hit Rate', 'N']
hit_rate['Hit Rate'] = (hit_rate['Hit Rate'] * 100).round(1)
hit_rate = hit_rate.sort_values('Hit Rate', ascending=False)

print("Profile → Actual 2nd-Contract Rate:")
print(hit_rate.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = {
    'Clean Hit': 'steelblue',
    'Hidden Gem': 'seagreen',
    'Hedge Risk': 'goldenrod',
    'Grade-Text Mismatch (Overrated)': 'tomato',
    'Unclassified': 'lightgray',
}
bars = ax.barh(
    hit_rate.index,
    hit_rate['Hit Rate'],
    color=[colors.get(p, 'gray') for p in hit_rate.index],
    edgecolor='white',
    height=0.6,
)
for bar, (_, row) in zip(bars, hit_rate.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{row['Hit Rate']:.1f}%  (n={int(row['N'])})",
            va='center', fontsize=9)
ax.set_xlabel('2nd Contract Rate (%)')
ax.set_title('Mismatch Profile → Actual Success Rate (2014–2021)', fontweight='bold')
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig('profile_hit_rates.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Hidden Gem examples ───────────────────────────────────────────────────────
gem_cols = ['year', 'player_name', 'Pos_Group', 'grade', 'sim_hit',
            'technique_density', 'made_it_contract']
gems = (pos_df[pos_df['profile'] == 'Hidden Gem']
        .sort_values('sim_hit', ascending=False)
        [gem_cols]
        .head(15))
gems.columns = ['Year', 'Player', 'Pos', 'Grade', 'Sim→Success', 'Tech Density', 'Made It']
gems['Sim→Success'] = gems['Sim→Success'].round(4)
gems['Tech Density'] = gems['Tech Density'].round(4)
print("Top Hidden Gems (low grade, high technique density, high success centroid similarity):")
print(gems.to_string(index=False))


In [ ]:
# ── Overrated examples ────────────────────────────────────────────────────────
over_cols = ['year', 'player_name', 'Pos_Group', 'grade', 'sim_hit',
             'raw_athlete_density', 'made_it_contract']
overrated = (pos_df[pos_df['profile'] == 'Grade-Text Mismatch (Overrated)']
             .sort_values(['grade', 'sim_hit'], ascending=[False, True])
             [over_cols]
             .head(15))
overrated.columns = ['Year', 'Player', 'Pos', 'Grade', 'Sim→Success', 'Raw Athlete Density', 'Made It']
overrated['Sim→Success'] = overrated['Sim→Success'].round(4)
overrated['Raw Athlete Density'] = overrated['Raw Athlete Density'].round(4)
print("Top Overrated Players (high grade, text diverges from Success Centroid):")
print(overrated.to_string(index=False))


## Section 6 — Predictive Power Analysis

**Two logistic regression models compared via 5-fold cross-validated AUC:**

| Model | Features |
|-------|---------|
| **Baseline (Grade Only)** | `grade` |
| **Scouting Alpha (Grade + NLP)** | `grade` + `sim_hit` + `raw_athlete_density` + `technique_density` + `hedge_density` + `definitive_density` |

We also report the **coefficient magnitude** of each NLP feature to quantify which clusters
captured signal that grade alone missed.


In [ ]:
# ── Build modelling dataset ───────────────────────────────────────────────────
model_df = pos_df.dropna(subset=['sim_hit', 'grade', 'made_it_contract']).copy()
model_df['made_it_contract'] = model_df['made_it_contract'].astype(int)

FEATURE_COLS = [
    'grade',
    'sim_hit',
    'raw_athlete_density',
    'technique_density',
    'hedge_density',
    'definitive_density',
]
GRADE_ONLY = ['grade']

X_full  = model_df[FEATURE_COLS].values
X_grade = model_df[GRADE_ONLY].values
y       = model_df['made_it_contract'].values

print(f"Modelling set: {len(model_df)} players "
      f"({y.sum()} hits, {(1-y).sum()} busts, "
      f"hit rate {y.mean():.1%})")


In [ ]:
# ── Cross-validated AUC comparison ───────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(X, y, cv):
    """Return array of ROC-AUC scores across folds."""
    scaler = StandardScaler()
    lr = LogisticRegression(max_iter=2000, solver='lbfgs', class_weight='balanced', random_state=42)
    pipe = Pipeline([('scaler', scaler), ('lr', lr)])
    return cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')

auc_full  = cv_auc(X_full,  y, cv)
auc_grade = cv_auc(X_grade, y, cv)

print("=" * 55)
print(f"  Grade-Only Model AUC:          {auc_grade.mean():.4f} ± {auc_grade.std():.4f}")
print(f"  Scouting Alpha (Grade+NLP) AUC: {auc_full.mean():.4f} ± {auc_full.std():.4f}")
print(f"  Delta:                         {auc_full.mean() - auc_grade.mean():+.4f}")
print("=" * 55)
print(f"\nFold-level AUC — Grade only: {[f'{s:.3f}' for s in auc_grade]}")
print(f"Fold-level AUC — Full model: {[f'{s:.3f}' for s in auc_full]}")


In [ ]:
# ── Feature coefficients ──────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)
lr_full = LogisticRegression(max_iter=2000, solver='lbfgs', class_weight='balanced', random_state=42)
lr_full.fit(X_scaled, y)

coef_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Coefficient': lr_full.coef_[0],
}).sort_values('Coefficient', ascending=False)

print("\nLogistic Regression Coefficients (standardized features):")
print("Positive = associated with earning 2nd contract")
print("-" * 45)
print(coef_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized Coefficient')
ax.set_title('Scouting Alpha — Feature Importance\n(Logistic Regression, balanced classes)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-position AUC breakdown ────────────────────────────────────────────────
pos_auc_rows = []
for pos in TARGET_POSITIONS:
    sub = model_df[model_df['Pos_Group'] == pos]
    if len(sub) < 30 or sub['made_it_contract'].nunique() < 2:
        pos_auc_rows.append({'Position': pos, 'Grade AUC': np.nan, 'Alpha AUC': np.nan, 'N': len(sub)})
        continue

    y_pos = sub['made_it_contract'].values
    folds = min(5, max(2, int(y_pos.sum() // 5)))
    cv_pos = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    try:
        auc_g = cv_auc(sub[GRADE_ONLY].values, y_pos, cv_pos)
        auc_a = cv_auc(sub[FEATURE_COLS].values, y_pos, cv_pos)
        pos_auc_rows.append({
            'Position': pos,
            'Grade AUC': round(auc_g.mean(), 4),
            'Alpha AUC': round(auc_a.mean(), 4),
            'N': len(sub),
            'Delta': round(auc_a.mean() - auc_g.mean(), 4),
        })
    except Exception as e:
        pos_auc_rows.append({'Position': pos, 'Grade AUC': np.nan, 'Alpha AUC': np.nan, 'N': len(sub), 'Delta': np.nan})

pos_auc_df = pd.DataFrame(pos_auc_rows).dropna(subset=['Grade AUC'])
print("\nPer-Position AUC: Grade-Only vs. Scouting Alpha")
print(pos_auc_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(pos_auc_df))
w = 0.35
ax.bar(x - w/2, pos_auc_df['Grade AUC'], w, label='Grade Only', color='lightcoral')
ax.bar(x + w/2, pos_auc_df['Alpha AUC'], w, label='Grade + NLP Alpha', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(pos_auc_df['Position'])
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Random baseline')
ax.legend()
ax.set_title('ROC-AUC by Position: Grade-Only vs. Scouting Alpha', fontweight='bold')
plt.tight_layout()
plt.savefig('auc_by_position.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Which NLP cluster adds the most signal grade misses? ──────────────────────
# Strategy: add features one-at-a-time to grade and measure AUC lift

incremental_results = []
for feat in FEATURE_COLS[1:]:  # skip grade itself
    X_inc = model_df[['grade', feat]].values
    auc_inc = cv_auc(X_inc, y, cv)
    incremental_results.append({
        'NLP Feature Added': feat,
        'AUC (Grade + Feature)': round(auc_inc.mean(), 4),
        'Lift over Grade-Only': round(auc_inc.mean() - auc_grade.mean(), 4),
    })

inc_df = pd.DataFrame(incremental_results).sort_values('Lift over Grade-Only', ascending=False)
print("\nIncremental AUC Lift from adding each NLP feature to grade:")
print(inc_df.to_string(index=False))
print(f"\n→ Best single NLP addition: '{inc_df.iloc[0]['NLP Feature Added']}' "
      f"(+{inc_df.iloc[0]['Lift over Grade-Only']:.4f} AUC)")


## Section 7 — Output Summary

In [ ]:
# ── Master output table ───────────────────────────────────────────────────────
print("=" * 80)
print("SCOUTING ALPHA — POSITION GROUP SUMMARY")
print("=" * 80)

for pos in TARGET_POSITIONS:
    if pos not in pos_results:
        continue

    sub = pos_df[pos_df['Pos_Group'] == pos]
    pos_anchor = anchor_df[anchor_df['pos_group'] == pos].sort_values('rank')

    success_words = ', '.join(pos_anchor['success_word'].tolist())
    bust_words    = ', '.join(pos_anchor['bust_word'].tolist())

    hidden_gems = (sub[sub['profile'] == 'Hidden Gem']
                   .sort_values('sim_hit', ascending=False)
                   [['player_name', 'year', 'grade', 'sim_hit', 'made_it_contract']]
                   .head(3))

    overrated = (sub[sub['profile'] == 'Grade-Text Mismatch (Overrated)']
                 .sort_values(['grade', 'sim_hit'], ascending=[False, True])
                 [['player_name', 'year', 'grade', 'sim_hit', 'made_it_contract']]
                 .head(3))

    pos_auc_row = pos_auc_df[pos_auc_df['Position'] == pos]
    auc_str = (f"Grade AUC={pos_auc_row['Grade AUC'].values[0]:.3f}  "
               f"Alpha AUC={pos_auc_row['Alpha AUC'].values[0]:.3f}  "
               f"Δ={pos_auc_row['Delta'].values[0]:+.3f}"
               if len(pos_auc_row) > 0 else "N/A")

    print(f"\n{'─'*70}")
    print(f"  POSITION: {pos}  |  n={len(sub)}  |  Hit Rate={sub['made_it_contract'].mean():.1%}")
    print(f"  AUC: {auc_str}")
    print(f"  Top 5 SUCCESS words: {success_words}")
    print(f"  Top 5 BUST words:    {bust_words}")

    if not hidden_gems.empty:
        print(f"\n  💎 Hidden Gems:")
        for _, row in hidden_gems.iterrows():
            outcome = '✅ Made it' if row['made_it_contract'] == 1 else '❌ Did not'
            print(f"      {row['player_name']:30s} {row['year']}  grade={row['grade']:.1f}  sim={row['sim_hit']:.3f}  {outcome}")

    if not overrated.empty:
        print(f"\n  ⚠️  Overrated (high grade, text misaligned):")
        for _, row in overrated.iterrows():
            outcome = '✅ Made it' if row['made_it_contract'] == 1 else '❌ Did not'
            print(f"      {row['player_name']:30s} {row['year']}  grade={row['grade']:.1f}  sim={row['sim_hit']:.3f}  {outcome}")

print(f"\n{'='*80}")


In [ ]:
# ── Full player-level output CSV ──────────────────────────────────────────────
output_cols = [
    'year', 'player_name', 'Pos_Group', 'grade',
    'sim_hit', 'sim_bust',
    'raw_athlete_density', 'technique_density', 'hedge_density', 'definitive_density',
    'profile', 'made_it_contract',
]
output_df = pos_df[output_cols].copy()
output_df = output_df.rename(columns={
    'Pos_Group': 'pos_group',
    'sim_hit':   'cosine_sim_success_centroid',
    'sim_bust':  'cosine_sim_bust_centroid',
})

output_df.to_csv('scouting_alpha_player_profiles.csv', index=False)
print(f"Player-level output saved: scouting_alpha_player_profiles.csv")
print(f"Shape: {output_df.shape}")
print(f"\nProfile distribution:")
print(output_df['profile'].value_counts().to_string())
print(f"\nSample rows:")
print(output_df.head(5).to_string(index=False))


In [ ]:
# ── Anchor word output CSV ────────────────────────────────────────────────────
anchor_df.to_csv('scouting_alpha_anchor_words.csv', index=False)
print("Anchor word table saved: scouting_alpha_anchor_words.csv")
print(anchor_df.to_string(index=False))


---
## Schema Reference

If you need to plug in new data, the pipeline expects a DataFrame with these columns:

| Column | Type | Description |
|--------|------|-------------|
| `year` | int | Draft year (filter to 2014–2021 for training) |
| `player_name` | str | Player full name |
| `Pos_Group` | str | Position group: EDGE, OL, WR, QB, RB, DB |
| `grade` | float | Scout's numerical grade (typical range 5.0–7.5) |
| `overview` | str | Scout report overview section |
| `strengths` | str | Scout report strengths section |
| `weaknesses` | str | Scout report weaknesses section |
| `made_it_contract` | int | 1 = earned 2nd contract, 0 = did not |

The three text columns are concatenated into `scouting_text` and preprocessed via `nfl_preprocess()`.
Update `DATA_PATH` at the top of Section 1 to point at your data file.
